# Diffusion Planner: executable architecture derivation

Every implementation cell below defines the actual class or function used by `src/diffusion_planner`. Run cells in order: dependencies are intentionally introduced before their users. The default dimensions are the official released-checkpoint dimensions.


## Full architecture pipeline

### ARCHITECTURE MAP

```text
scene tensors -> structured encoders -> scene tokens + position tokens -> masked fusion -> context [B,107,192]
route lanes -> RouteEncoder --+
timestep -> sinusoid + MLP ----+-> condition [B,192] -> DiT x3(context cross-attention) -> x_start [B,11,324]
clean futures -> normalization -> VP-SDE future noising + fixed current state -> x_start losses
```

### MODULE I/O CARD INDEX

| MODULE I/O CARD | Input -> transform -> output |
| --- | --- |
| Configuration / embeddings / DropPath | metadata or `(B,...)` -> lookup, sinusoid, stochastic residual -> shape preserved or `(B,192)` |
| VP-SDE / normalizer | future `(B,11,80,4)` -> z-score or marginal noising -> same shape |
| MLP / Mixer | `(N,C)` or `(N,T,C)` -> channel/token mixing -> same shape |
| Agent / static / lane encoders | raw scene tensors -> valid-token encoding plus masks/positions -> `(B,N,192)` |
| Fusion / route encoder | scene tokens or `(B,25,20,12)` -> attention or pool -> `(B,107,192)` or `(B,192)` |
| DiT / final layer | noisy `(B,11,324)`, context, condition -> denoising -> `(B,11,324)` |
| Planner loss / guidance | denoised trajectories -> masked objective or energy -> scalar losses |
```text
Every following section expands its unit's Dimension evolution note into this input -> transform -> output pattern.
```

```text
CONDITIONING INPUTS
===================
neighbor_agents_past  (B, 32, 21, 11) -- AgentFusionEncoder -- (B, 32, 192) --+
static_objects        (B,  5, 10)     -- StaticFusionEncoder - (B,  5, 192) --+
lanes                 (B, 70, 20, 12) -- LaneFusionEncoder -- (B, 70, 192) --+
                                                                                |
position features     (B,107,  7)     -- Linear(7,192) ------------------------+
                                                                                v
                                            masked FusionEncoder x3
                                      context encoding: (B, 107, 192)
                                                                                |
                                                                                | cross-attention keys / values
                                                                                v
NOISY JOINT TRAJECTORY                                                     DiT x3
======================                                      +-------------------------------+
x_t (B, 11, 324) -> MLP(324,192) + ego/neighbor identity -> | self-attention + adaLN MLP    |
                                                        ^     | cross-attention(context)       |
                                                        |     | MLP                           |
                                                        |     +-------------------------------+
                                                        |                     |
NAVIGATION + TIME                                      |                     v
=================                                      |          adaLN FinalLayer
route_lanes (B,25,20,12) -> RouteEncoder -> (B,192) --+--> condition (B,192) -> x_start (B,11,324)
diffusion t (B,) -> sinusoid + MLP -> (B,192) -------+

TRAINING TARGET
===============
ego future (B,80,4) + neighbor future (B,10,80,4)
 -> StateNormalizer -> prepend unnoised current state -> VP-SDE noising of future only
 -> x_start MSE: ego planning loss + valid neighbor prediction loss
```

`11 = 1 ego + 10 predicted neighbors`; `324 = (1 current + 80 future) * 4`, where each state is `(x, y, cos(yaw), sin(yaw))`. The context path models scene entities, while the route-plus-time path controls every adaLN-modulated DiT block.


## Mathematical formulation

Let $B$ be batch size, $P=11$ the ego-plus-neighbor participant count, $T=80$ future steps, $D=192$ hidden width, and $\mathbf{x}_0 \in \mathbb{R}^{B \times P \times T \times 4}$ normalized clean future states. The implementation uses the following equations.

### State normalization and VP-SDE

For state mean $\boldsymbol{\mu}=(10,0,0,0)$ and standard deviation $\boldsymbol{\sigma}=(20,20,1,1)$,

$$\bar{\mathbf{x}}_0 = (\mathbf{x}_0 - \boldsymbol{\mu}) \oslash \boldsymbol{\sigma}.$$,

The linear variance-preserving schedule has $\beta(t)=\beta_{\min}+t(\beta_{\max}-\beta_{\min})$, with $\beta_{\min}=0.1$ and $\beta_{\max}=20$. The closed-form forward process implemented by `VPSchedule.marginal_prob` is

$$\log \alpha_t = -\frac{1}{4}t^2(\beta_{\max}-\beta_{\min}) - \frac{1}{2}\beta_{\min}t,$$
$$q_t(\mathbf{x}_t \mid \bar{\mathbf{x}}_0)=\mathcal{N}(\alpha_t\bar{\mathbf{x}}_0,\sigma_t^2\mathbf{I}), \qquad \sigma_t=\sqrt{1-\alpha_t^2},$$
$$\mathbf{x}_t=\alpha_t\bar{\mathbf{x}}_0+\sigma_t\boldsymbol{\epsilon}, \qquad \boldsymbol{\epsilon}\sim\mathcal{N}(0,\mathbf{I}).$$

Only the future suffix is noised. The observed current state $\mathbf{s}^{0}$ is prepended unchanged: $[\mathbf{s}^{0},\mathbf{x}_t]$.

### Vector encoders and context fusion

Each two-layer MLP is $\operatorname{MLP}(\mathbf{z})=\mathbf{W}_2\operatorname{GELU}(\mathbf{W}_1\mathbf{z}+\mathbf{b}_1)+\mathbf{b}_2$. For an MLP-Mixer input $\mathbf{S}\in\mathbb{R}^{N\times C}$, the implementation applies

$$\mathbf{S}\leftarrow\mathbf{S}+\operatorname{MLP}_{token}(\operatorname{LN}(\mathbf{S})^\top)^\top,$$
$$\mathbf{S}\leftarrow\mathbf{S}+\operatorname{MLP}_{channel}(\operatorname{LN}(\mathbf{S})).$$

Agent histories and lane polylines are mixed and mean pooled. Their semantic embeddings are added before the output projection: agent type $\mathbf{e}_{type}$; lane speed-limit $\mathbf{e}_{speed}$; and traffic-light $\mathbf{e}_{traffic}$. The 107 scene tokens receive projected pose/modality features, then masked self-attention:


$$\operatorname{Attn}(\mathbf{Q},\mathbf{K},\mathbf{V})=\operatorname{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^\top}{\sqrt{d_k}}+\mathbf{M}\right)\mathbf{V},$$

where $\mathbf{M}$ assigns $-\infty$ to padded keys.

### Route/time conditioning and DiT

Route lanes are mixed into $\mathbf{r}\in\mathbb{R}^{B\times D}$. The sinusoidal timestep embedding is projected into $\mathbf{e}_t\in\mathbb{R}^{B\times D}$, and every DiT block receives

$$\mathbf{c}=\mathbf{r}+\mathbf{e}_t.$$,

Adaptive LayerNorm uses modulation parameters from $[\boldsymbol{\delta},\boldsymbol{\gamma},\mathbf{g}]=\operatorname{Linear}(\operatorname{SiLU}(\mathbf{c}))$:

$$\operatorname{adaLN}(\mathbf{h},\boldsymbol{\delta},\boldsymbol{\gamma})=\operatorname{LN}(\mathbf{h})\odot(1+\boldsymbol{\gamma})+\boldsymbol{\delta}.$$

The released DiT self-attention and first MLP updates are

$$\mathbf{h}\leftarrow\mathbf{h}+\mathbf{g}_{msa}\odot\operatorname{MHA}(\operatorname{adaLN}(\mathbf{h})),$$
$$\mathbf{h}\leftarrow\mathbf{h}+\mathbf{g}_{mlp}\odot\operatorname{MLP}(\operatorname{adaLN}(\mathbf{h})).$$

It then applies cross-attention against scene context and the final adaLN projection to estimate $\hat{\mathbf{x}}_0=\mu_\theta(\mathbf{x}_t,t,\mathbf{C})$.

### Released `x_start` objective

With ego target index $0$, valid-neighbor indicator $v_{p,\tau}$, and future-state dimensions summed by $\|\cdot\|_2^2$, the implementation reports

$$\mathcal{L}_{ego}=\frac{1}{T}\sum_{\tau=1}^{T}\|\hat{\mathbf{x}}_{0,\tau}-\bar{\mathbf{x}}_{0,\tau}\|_2^2,$$
$$\mathcal{L}_{neighbor}=\frac{1}{\sum_{p>0,\tau}v_{p,\tau}}\sum_{p=1}^{P-1}\sum_{\tau=1}^{T}v_{p,\tau}\|\hat{\mathbf{x}}_{p,\tau}-\bar{\mathbf{x}}_{p,\tau}\|_2^2.$$

These are `ego_planning_loss` and `neighbor_prediction_loss` returned by `DiffusionPlanner.diffusion_loss`.


In [ ]:
import math
from copy import copy

import torch
from torch import nn

torch.manual_seed(42)
device = torch.device("cpu")  # Offline synthetic default

## Preparing future labels

Upstream futures carry `(x, y, yaw)` in the ego frame, with positions in meters and yaw in radians. Convert yaw to cosine and sine. Identify missing neighbor rows before conversion, because `cos(0) = 1` would otherwise turn a padded row into a nonzero vector. The ego future is never treated as padding.


In [ ]:
raw_neighbors = torch.tensor([[[0.0, 0.0, 0.0], [1.0, 2.0, torch.pi / 2]]])
missing = ~raw_neighbors.ne(0).any(dim=-1)
encoded_neighbors = torch.cat(
    (raw_neighbors[..., :2], raw_neighbors[..., 2:3].cos(), raw_neighbors[..., 2:3].sin()), dim=-1
)
encoded_neighbors[missing] = 0
print("encoded labels:", encoded_neighbors, "missing:", missing)

## 1. Official checkpoint configuration

```text
args.json -> PlannerConfig
21 history, 80 future, 32 context agents, 10 predicted neighbors
70 lane polylines, 25 route polylines, hidden width 192
```

This dataclass fixes every architectural dimension. These are the released checkpoint defaults; small dimensions are used only in tests.

**Dimension evolution:** I/O: no tensors. Output: immutable model-shape contract.


In [ ]:
class PlannerConfig:
    """Defaults from the official released checkpoint ``args.json``."""

    future_len: int = 80
    time_len: int = 21
    agent_state_dim: int = 11
    agent_num: int = 32
    static_objects_state_dim: int = 10
    static_objects_num: int = 5
    lane_len: int = 20
    lane_state_dim: int = 12
    lane_num: int = 70
    route_len: int = 20
    route_state_dim: int = 12
    route_num: int = 25
    predicted_neighbor_num: int = 10
    hidden_dim: int = 192
    num_heads: int = 6
    encoder_depth: int = 3
    decoder_depth: int = 3
    encoder_drop_path_rate: float = 0.1
    decoder_drop_path_rate: float = 0.1
    diffusion_model_type: str = "x_start"

    @property
    def participant_num(self) -> int:
        return 1 + self.predicted_neighbor_num

    @property
    def trajectory_dim(self) -> int:
        return (self.future_len + 1) * 4

## 2. Two-layer MLP

```text
(..., in) -> Linear(in, hidden) -> GELU -> Linear(hidden, out)
```

The official source uses `timm.Mlp`. This local implementation is mathematically equivalent.

**Dimension evolution:** I/O: `(..., in_features)` -> `(..., out_features)`.


In [ ]:
class Mlp(nn.Module):
    """Equivalent to timm's two-layer MLP used in the official source."""

    def __init__(
        self,
        in_features: int,
        hidden_features: int,
        out_features: int | None = None,
        drop: float = 0.0,
        approximate: str = "none",
    ) -> None:
        super().__init__()
        out_features = out_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = nn.GELU(approximate=approximate)
        self.drop1 = nn.Dropout(drop)
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop2 = nn.Dropout(drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.drop2(self.fc2(self.drop1(self.act(self.fc1(x)))))

## 3. Stochastic depth

```text
training: residual branch * Bernoulli(keep) / keep
eval:     residual branch unchanged
```

DropPath randomly removes complete residual branches per sample during training, matching the official `timm.DropPath` use.

**Dimension evolution:** I/O: tensor shape is preserved.


In [ ]:
class DropPath(nn.Module):
    """Per-sample stochastic depth, matching timm DropPath semantics."""

    def __init__(self, probability: float) -> None:
        super().__init__()
        self.probability = probability

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.probability == 0.0 or not self.training:
            return x
        keep = 1 - self.probability
        shape = (x.shape[0],) + (1,) * (x.ndim - 1)
        mask = x.new_empty(shape).bernoulli_(keep)
        if keep > 0:
            mask.div_(keep)
        return x * mask

## 4. MLP-Mixer block

```text
(B, tokens, channels)
  -> LayerNorm -> transpose -> token MLP -> transpose -> residual
  -> LayerNorm -> channel MLP -> residual
```

Sparse vector sequences need interaction across both points and feature channels. This is the exact mixer ordering used by the agent, lane, and route encoders.

**Dimension evolution:** I/O: `(B, tokens, channels)` -> `(B, tokens, channels)`.


In [ ]:
class MixerBlock(nn.Module):
    """Official token-mixing then channel-mixing residual block."""

    def __init__(self, tokens_mlp_dim: int, channels_mlp_dim: int, drop_path_rate: float) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(channels_mlp_dim)
        self.channels_mlp = Mlp(channels_mlp_dim, channels_mlp_dim, drop=drop_path_rate)
        self.norm2 = nn.LayerNorm(channels_mlp_dim)
        self.tokens_mlp = Mlp(tokens_mlp_dim, tokens_mlp_dim, drop=drop_path_rate)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.tokens_mlp(self.norm1(x).transpose(1, 2)).transpose(1, 2)
        x = x + y
        return x + self.channels_mlp(self.norm2(x))

## 5. Continuous VP-SDE

```text
x0, t -> mean=alpha(t)*x0, std=sigma(t)
xt = mean + std*epsilon
```

The released model uses the linear variance-preserving SDE with beta from 0.1 to 20.0. Training samples one continuous diffusion time for each scene.

**Dimension evolution:** I/O: `x0 (B, ...)`, `t (B,)` -> mean/std broadcast over `...`.


In [ ]:
class VPSchedule:
    def __init__(self, beta_min: float = 0.1, beta_max: float = 20.0) -> None:
        self.beta_min = beta_min
        self.beta_max = beta_max

    def marginal_prob(
        self, x: torch.Tensor, time: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor]:
        shape = (-1,) + (1,) * (x.ndim - 1)
        time = time.reshape(shape)
        mean_log_coefficient = (
            -0.25 * time.square() * (self.beta_max - self.beta_min) - 0.5 * self.beta_min * time
        )
        return mean_log_coefficient.exp() * x, (1 - (2 * mean_log_coefficient).exp()).sqrt()

    def std(self, time: torch.Tensor) -> torch.Tensor:
        return (
            1
            - torch.exp(
                -0.5 * time.square() * (self.beta_max - self.beta_min) - self.beta_min * time
            )
        ).sqrt()

    def add_noise(
        self, clean: torch.Tensor, time: torch.Tensor, noise: torch.Tensor | None = None
    ) -> torch.Tensor:
        mean, std = self.marginal_prob(clean, time)
        return mean + std * (torch.randn_like(clean) if noise is None else noise)

## 6. Trajectory state normalization

```text
state (x,y,cos,sin) -> (state - [10,0,0,0]) / [20,20,1,1]
```

The official checkpoint normalizes future ego and neighbor trajectories with these released statistics. The current state remains unnoised.

**Dimension evolution:** I/O: `(B, 1+Pn, T, 4)` -> same shape.


In [ ]:
class StateNormalizer:
    def __init__(self, participant_num: int) -> None:
        mean = torch.tensor([10.0, 0.0, 0.0, 0.0]).view(1, 1, 1, 4)
        std = torch.tensor([20.0, 20.0, 1.0, 1.0]).view(1, 1, 1, 4)
        self.mean = mean.expand(1, participant_num, 1, 4).clone()
        self.std = std.expand(1, participant_num, 1, 4).clone()

    def __call__(self, data: torch.Tensor) -> torch.Tensor:
        return (data - self.mean.to(data.device)) / self.std.to(data.device)

    def inverse(self, data: torch.Tensor) -> torch.Tensor:
        return data * self.std.to(data.device) + self.mean.to(data.device)

## 7. Observation normalization

```text
each feature tensor -> z-score valid entries -> preserve zero padding
```

This helper applies released preprocessing statistics without turning padding into nonzero values.

**Dimension evolution:** I/O: dictionary of feature tensors -> same keys and shapes.


In [ ]:
class ObservationNormalizer:
    """Apply official means/stds while preserving all-zero padded vectors."""

    def __init__(self, statistics: dict[str, tuple[list[float], list[float]]]) -> None:
        self.statistics = statistics

    def __call__(self, data: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
        output = copy(data)
        for name, (mean, std) in self.statistics.items():
            if name not in data:
                continue
            value = data[name]
            mask = value.ne(0).any(dim=-1, keepdim=True)
            output[name] = (value - torch.tensor(mean, device=value.device)) / torch.tensor(
                std, device=value.device
            )
            output[name] = output[name] * mask
        return output

## 8. Context self-attention block

```text
tokens -> LayerNorm -> masked MHA -> residual
       -> LayerNorm -> MLP        -> residual
```

This is the repeated transformer unit used after all scene entities become context tokens.

**Dimension evolution:** I/O: tokens `(B, N, 192)`, mask `(B, N)` -> tokens `(B, N, 192)`.


In [ ]:
class SelfAttentionBlock(nn.Module):
    def __init__(self, dim: int, heads: int, dropout: float) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout, batch_first=True)
        self.drop_path = DropPath(dropout) if dropout > 0 else nn.Identity()
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = Mlp(dim, dim * 4, drop=dropout)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        x = x + self.drop_path(self.attn(self.norm1(x), x, x, key_padding_mask=mask)[0])
        return x + self.drop_path(self.mlp(self.norm2(x)))

## 9. Neighbor-agent history encoder

```text
(B,32,21,11) -> state[8] + availability -> 9->128 MLP
-> temporal MLP 21->64 -> Mixer x3 -> mean pool
+ type embedding 3->128 -> 128->192 MLP
```

Eleven input features are pose, velocity, dimensions, and a vehicle/pedestrian/bicycle one-hot type. All-zero histories are masked.

**Dimension evolution:** I/O: `(B, 32, 21, 11)` -> tokens `(B, 32, 192)`, mask `(B,32)`, positions `(B,32,7)`.


In [ ]:
class AgentFusionEncoder(nn.Module):
    """Encode `(x,y,cos,sin,vx,vy,w,l,type[3])` histories per neighbor."""

    def __init__(self, config: PlannerConfig) -> None:
        super().__init__()
        self.type_emb = nn.Linear(3, 128)
        self.channel_pre_project = Mlp(9, 128, 128)
        self.token_pre_project = Mlp(config.time_len, 64, 64)
        self.blocks = nn.ModuleList(
            MixerBlock(64, 128, config.encoder_drop_path_rate) for _ in range(config.encoder_depth)
        )
        self.norm = nn.LayerNorm(128)
        self.emb_project = Mlp(
            128, config.hidden_dim, config.hidden_dim, drop=config.encoder_drop_path_rate
        )

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        batch, agents, frames, _ = x.shape
        types = x[:, :, -1, 8:]
        states = x[..., :8]
        pos = states[:, :, -1, :7].clone()
        pos[..., -3:] = 0
        pos[..., -3] = 1
        vector_mask = states.abs().sum(dim=-1) == 0
        token_mask = (~vector_mask).sum(dim=-1) == 0
        states = torch.cat([states, (~vector_mask).float().unsqueeze(-1)], dim=-1).reshape(
            batch * agents, frames, 9
        )
        valid = ~token_mask.reshape(-1)
        result = states.new_zeros(batch * agents, self.emb_project.fc2.out_features)
        if valid.any():
            encoded = self.channel_pre_project(states[valid]).transpose(1, 2)
            encoded = self.token_pre_project(encoded).transpose(1, 2)
            for block in self.blocks:
                encoded = block(encoded)
            encoded = encoded.mean(dim=1) + self.type_emb(types.reshape(batch * agents, 3)[valid])
            result[valid] = self.emb_project(self.norm(encoded))
        return result.reshape(batch, agents, -1), token_mask, pos

## 10. Static-object encoder

```text
(B,5,10) -> MLP 10->192
```

Static obstacles do not have a temporal polyline, so the official architecture directly projects each ten-feature vector and marks its modality in the position encoding.

**Dimension evolution:** I/O: `(B, 5, 10)` -> tokens `(B,5,192)`, mask `(B,5)`, positions `(B,5,7)`.


In [ ]:
class StaticFusionEncoder(nn.Module):
    def __init__(self, config: PlannerConfig) -> None:
        super().__init__()
        self.hidden_dim = config.hidden_dim
        self.projection = Mlp(
            config.static_objects_state_dim,
            config.hidden_dim,
            config.hidden_dim,
            drop=config.encoder_drop_path_rate,
        )

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        batch, objects, _ = x.shape
        pos = x[:, :, :7].clone()
        pos[..., -3:] = 0
        pos[..., -2] = 1
        mask = x[..., :10].abs().sum(dim=-1) == 0
        valid = ~mask.reshape(-1)
        result = x.new_zeros(batch * objects, self.hidden_dim)
        if valid.any():
            result[valid] = self.projection(x.reshape(batch * objects, -1)[valid])
        return result.reshape(batch, objects, -1), mask, pos

## 11. Lane encoder

```text
geometry[8] -> 8->128 MLP -> point MLP 20->64 -> Mixer x3 -> pool
+ speed-limit embedding + traffic-light embedding -> 128->192 MLP
```

Lane geometry, speed-limit availability, and four-state traffic-light features are explicitly separate signals in the official architecture.

**Dimension evolution:** I/O: lanes `(B,70,20,12)` plus limits `(B,70,1)` -> `(B,70,192)`, mask `(B,70)`, positions `(B,70,7)`.


In [ ]:
class LaneFusionEncoder(nn.Module):
    """Encode lane geometry, speed-limit availability, and traffic-light state."""

    def __init__(self, config: PlannerConfig) -> None:
        super().__init__()
        self.lane_len = config.lane_len
        self.speed_limit_emb = nn.Linear(1, 128)
        self.unknown_speed_emb = nn.Embedding(1, 128)
        self.traffic_emb = nn.Linear(4, 128)
        self.channel_pre_project = Mlp(8, 128, 128)
        self.token_pre_project = Mlp(config.lane_len, 64, 64)
        self.blocks = nn.ModuleList(
            MixerBlock(64, 128, config.encoder_drop_path_rate) for _ in range(config.encoder_depth)
        )
        self.norm = nn.LayerNorm(128)
        self.emb_project = Mlp(
            128, config.hidden_dim, config.hidden_dim, drop=config.encoder_drop_path_rate
        )

    def forward(
        self, x: torch.Tensor, speed_limit: torch.Tensor, has_speed_limit: torch.Tensor
    ) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        batch, lanes, points, _ = x.shape
        traffic = x[:, :, 0, 8:]
        states = x[..., :8]
        pos = states[:, :, self.lane_len // 2, :7].clone()
        heading = torch.atan2(pos[..., 3], pos[..., 2])
        pos[..., 2], pos[..., 3] = heading.cos(), heading.sin()
        pos[..., -3:] = 0
        pos[..., -1] = 1
        vector_mask = states.abs().sum(dim=-1) == 0
        token_mask = (~vector_mask).sum(dim=-1) == 0
        valid = ~token_mask.reshape(-1)
        result = states.new_zeros(batch * lanes, self.emb_project.fc2.out_features)
        if valid.any():
            encoded = self.channel_pre_project(
                states.reshape(batch * lanes, points, 8)[valid]
            ).transpose(1, 2)
            encoded = self.token_pre_project(encoded).transpose(1, 2)
            for block in self.blocks:
                encoded = block(encoded)
            encoded = encoded.mean(dim=1)
            speed = speed_limit.reshape(batch * lanes, 1)[valid]
            known = has_speed_limit.reshape(batch * lanes)[valid].bool()
            speed_embedding = encoded.new_zeros(encoded.shape)
            if known.any():
                speed_embedding[known] = self.speed_limit_emb(speed[known])
            if (~known).any():
                speed_embedding[~known] = self.unknown_speed_emb.weight.expand((~known).sum(), -1)
            encoded = (
                encoded
                + speed_embedding
                + self.traffic_emb(traffic.reshape(batch * lanes, 4)[valid])
            )
            result[valid] = self.emb_project(self.norm(encoded))
        return result.reshape(batch, lanes, -1), token_mask, pos

## 12. Scene-token fusion

```text
32 agent + 5 static + 70 lane = 107 tokens
-> masked self-attention block x3 -> LayerNorm
```

The fusion transformer enables each entity to use the whole structured scene. The official implementation forces token zero to remain unmasked.

**Dimension evolution:** I/O: `(B,107,192)`, `(B,107)` -> `(B,107,192)`.


In [ ]:
class FusionEncoder(nn.Module):
    def __init__(self, config: PlannerConfig) -> None:
        super().__init__()
        self.blocks = nn.ModuleList(
            SelfAttentionBlock(config.hidden_dim, config.num_heads, config.encoder_drop_path_rate)
            for _ in range(config.encoder_depth)
        )
        self.norm = nn.LayerNorm(config.hidden_dim)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        mask = mask.clone()
        mask[:, 0] = False
        for block in self.blocks:
            x = block(x, mask)
        return self.norm(x)

## 13. Full structured context encoder

```text
agent tokens + static tokens + lane tokens
+ position projection 7->192
-> FusionEncoder -> cross-attention context
```

This combines all non-route scene conditions used by the DiT decoder.

**Dimension evolution:** I/O: condition dictionary -> `encoding (B,107,192)`, `mask (B,107)`.


In [ ]:
class ContextEncoder(nn.Module):
    """Official concatenation of agent, static-object, and lane context tokens."""

    def __init__(self, config: PlannerConfig) -> None:
        super().__init__()
        self.config = config
        self.agents = AgentFusionEncoder(config)
        self.static = StaticFusionEncoder(config)
        self.lanes = LaneFusionEncoder(config)
        self.position = nn.Linear(7, config.hidden_dim)
        self.fusion = FusionEncoder(config)

    def forward(self, inputs: dict[str, torch.Tensor]) -> dict[str, torch.Tensor]:
        agents, agent_mask, agent_pos = self.agents(inputs["neighbor_agents_past"])
        static, static_mask, static_pos = self.static(inputs["static_objects"])
        lanes, lane_mask, lane_pos = self.lanes(
            inputs["lanes"], inputs["lanes_speed_limit"], inputs["lanes_has_speed_limit"]
        )
        tokens = torch.cat([agents, static, lanes], dim=1)  # I/O: 32 + 5 + 70 -> (B,107,192)
        mask = torch.cat([agent_mask, static_mask, lane_mask], dim=1)  # I/O: -> (B,107)
        positions = torch.cat([agent_pos, static_pos, lane_pos], dim=1)  # I/O: -> (B,107,7)
        tokens = tokens + self.position(positions) * (~mask).unsqueeze(
            -1
        )  # I/O: position (B,107,7) -> (B,107,192)
        return {
            "encoding": self.fusion(tokens, mask),
            "mask": mask,
        }  # I/O: context stays (B,107,192)

## 14. Adaptive LayerNorm modulation

```text
modulate(x, shift, scale) = x * (1 + scale) + shift
```

DiT uses this operation to inject the route-plus-timestep condition into normalized trajectory tokens.

**Dimension evolution:** I/O: `x (B,P,D)`, shift/scale `(B,D)` -> `(B,P,D)`.


In [ ]:
def modulate(x: torch.Tensor, shift: torch.Tensor, scale: torch.Tensor) -> torch.Tensor:
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

## 15. Diffusion-time embedding

```text
t (B,) -> 256-d sinusoid -> Linear -> SiLU -> Linear -> 192
```

The continuous noise level receives a sinusoidal embedding before being added to the route condition.

**Dimension evolution:** I/O: `(B,)` -> `(B,192)`.


In [ ]:
class TimestepEmbedder(nn.Module):
    def __init__(self, hidden_size: int, frequency_embedding_size: int = 256) -> None:
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(frequency_embedding_size, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
        )
        self.frequency_embedding_size = frequency_embedding_size

    @staticmethod
    def timestep_embedding(t: torch.Tensor, dim: int, max_period: int = 10000) -> torch.Tensor:
        half = dim // 2
        frequencies = torch.exp(
            -math.log(max_period) * torch.arange(half, dtype=torch.float32, device=t.device) / half
        )
        embedding = torch.cat(
            [
                (t[:, None].float() * frequencies[None]).cos(),
                (t[:, None].float() * frequencies[None]).sin(),
            ],
            dim=-1,
        )
        return (
            torch.cat([embedding, torch.zeros_like(embedding[:, :1])], dim=-1)
            if dim % 2
            else embedding
        )

    def forward(self, t: torch.Tensor) -> torch.Tensor:
        return self.mlp(self.timestep_embedding(t, self.frequency_embedding_size))

## 16. Official DiT block

```text
trajectory tokens -> adaLN self-attention -> gated residual
                  -> adaLN MLP            -> gated residual
                  -> cross-attention over 107 scene tokens
                  -> MLP
```

This reproduces the public release exactly, including its cross-attention assignment structure rather than replacing it with a conventional residual variant.

**Dimension evolution:** I/O: trajectories `(B,11,192)`, context `(B,107,192)`, condition `(B,192)` -> `(B,11,192)`.


In [ ]:
class DiTBlock(nn.Module):
    """Official adaLN-Zero self-attention/MLP followed by cross-attention/MLP."""

    def __init__(self, dim: int, heads: int, dropout: float, mlp_ratio: float = 4.0) -> None:
        super().__init__()
        hidden = int(dim * mlp_ratio)
        self.norm1 = nn.LayerNorm(dim)
        self.attn = nn.MultiheadAttention(dim, heads, dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp1 = Mlp(dim, hidden, approximate="tanh")
        self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(dim, 6 * dim))
        self.norm3 = nn.LayerNorm(dim)
        self.cross_attn = nn.MultiheadAttention(dim, heads, dropout, batch_first=True)
        self.norm4 = nn.LayerNorm(dim)
        self.mlp2 = Mlp(dim, hidden, approximate="tanh")

    def forward(
        self,
        x: torch.Tensor,
        cross_c: torch.Tensor,
        condition: torch.Tensor,
        attn_mask: torch.Tensor,
    ) -> torch.Tensor:
        shift_msa, scale_msa, gate_msa, shift_mlp, scale_mlp, gate_mlp = self.adaLN_modulation(
            condition
        ).chunk(6, dim=1)
        modulated = modulate(self.norm1(x), shift_msa, scale_msa)
        x = (
            x
            + gate_msa.unsqueeze(1)
            * self.attn(modulated, modulated, modulated, key_padding_mask=attn_mask)[0]
        )
        x = x + gate_mlp.unsqueeze(1) * self.mlp1(modulate(self.norm2(x), shift_mlp, scale_mlp))
        # These two assignments intentionally match the official released implementation.
        x = self.cross_attn(self.norm3(x), cross_c, cross_c)[0]
        return self.mlp2(self.norm4(x))

## 17. adaLN output layer

```text
trajectory token -> adaLN(condition) -> 192->768->324 projection
```

The final condition-modulated projection returns all current-plus-future state values for one participant in one token.

**Dimension evolution:** I/O: `(B,11,192)` + `(B,192)` -> `(B,11,324)`.


In [ ]:
class FinalLayer(nn.Module):
    def __init__(self, hidden_size: int, output_size: int) -> None:
        super().__init__()
        self.norm_final = nn.LayerNorm(hidden_size)
        self.proj = nn.Sequential(
            nn.LayerNorm(hidden_size),
            nn.Linear(hidden_size, hidden_size * 4),
            nn.GELU(approximate="tanh"),
            nn.LayerNorm(hidden_size * 4),
            nn.Linear(hidden_size * 4, output_size),
        )
        self.adaLN_modulation = nn.Sequential(nn.SiLU(), nn.Linear(hidden_size, 2 * hidden_size))

    def forward(self, x: torch.Tensor, condition: torch.Tensor) -> torch.Tensor:
        shift, scale = self.adaLN_modulation(condition).chunk(2, dim=1)
        return self.proj(modulate(self.norm_final(x), shift, scale))

## 18. Navigation route encoder

```text
(B,25,20,12) -> first 4 geometry values -> flatten 500 points
4->64 MLP -> point MLP 500->32 -> Mixer -> pool -> 64->192 MLP
```

Route lanes are not concatenated to the context transformer. They produce a single navigation token that is added to every timestep embedding.

**Dimension evolution:** I/O: `(B,25,20,12)` -> route condition `(B,192)`.


In [ ]:
class RouteEncoder(nn.Module):
    """Encode all route-lane points into the navigation condition token."""

    def __init__(self, config: PlannerConfig) -> None:
        super().__init__()
        self.channel_pre_project = Mlp(4, 64, 64)
        self.token_pre_project = Mlp(config.route_num * config.lane_len, 32, 32)
        self.mixer = MixerBlock(32, 64, config.encoder_drop_path_rate)
        self.norm = nn.LayerNorm(64)
        self.emb_project = Mlp(
            64, config.hidden_dim, config.hidden_dim, drop=config.encoder_drop_path_rate
        )

    def forward(self, route_lanes: torch.Tensor) -> torch.Tensor:
        x = route_lanes[..., :4]
        batch, routes, points, _ = x.shape
        point_mask = x.abs().sum(dim=-1) == 0
        lane_mask = (~point_mask).sum(dim=-1) == 0
        valid = ~(lane_mask.sum(dim=-1) == routes)
        flattened = x.reshape(batch, routes * points, 4)
        result = x.new_zeros(batch, self.emb_project.fc2.out_features)
        if valid.any():
            encoded = self.channel_pre_project(flattened[valid]).transpose(1, 2)
            encoded = self.token_pre_project(encoded).transpose(1, 2)
            encoded = self.mixer(encoded).mean(dim=1)
            result[valid] = self.emb_project(self.norm(encoded))
        return result

## 19. Route-conditioned trajectory denoiser

```text
noisy (B,11,324) -> 324->192 MLP + ego/neighbor identity
route(192) + time(192) -> condition
DiT block x3 -> adaLN final -> clean/score (B,11,324)
```

One token jointly represents every state for each participant. Self-attention models ego-neighbor interactions; cross-attention retrieves scene context.

**Dimension evolution:** I/O: noisy `(B,11,324)`, scene context `(B,107,192)`, route `(B,25,20,12)` -> `(B,11,324)`.


In [ ]:
class TrajectoryDiT(nn.Module):
    """Official route-conditioned DiT trajectory model."""

    def __init__(self, config: PlannerConfig) -> None:
        super().__init__()
        self.model_type = config.diffusion_model_type
        if self.model_type not in {"score", "x_start"}:
            raise ValueError("diffusion_model_type must be 'score' or 'x_start'.")
        self.schedule = VPSchedule()
        self.route_encoder = RouteEncoder(config)
        self.agent_embedding = nn.Embedding(2, config.hidden_dim)
        self.preproj = Mlp(config.trajectory_dim, 512, config.hidden_dim)
        self.t_embedder = TimestepEmbedder(config.hidden_dim)
        self.blocks = nn.ModuleList(
            DiTBlock(config.hidden_dim, config.num_heads, config.decoder_drop_path_rate)
            for _ in range(config.decoder_depth)
        )
        self.final_layer = FinalLayer(config.hidden_dim, config.trajectory_dim)

    def forward(
        self,
        x: torch.Tensor,
        t: torch.Tensor,
        cross_c: torch.Tensor,
        route_lanes: torch.Tensor,
        neighbor_current_mask: torch.Tensor,
    ) -> torch.Tensor:
        batch, participants, _ = x.shape
        x = self.preproj(x)  # I/O: noisy (B,11,324) -> trajectory tokens (B,11,192)
        identities = torch.zeros(participants, dtype=torch.long, device=x.device)
        identities[1:] = 1
        x = (
            x + self.agent_embedding(identities)[None]
        )  # I/O: identity (11,192) broadcasts into (B,11,192)
        condition = self.route_encoder(route_lanes) + self.t_embedder(
            t
        )  # I/O: route (B,192) + time (B,192) -> (B,192)
        mask = torch.zeros(batch, participants, dtype=torch.bool, device=x.device)
        mask[:, 1:] = neighbor_current_mask
        for block in self.blocks:
            x = block(x, cross_c, condition, mask)
        x = self.final_layer(x, condition)  # I/O: (B,11,192) -> x_start/score (B,11,324)
        if self.model_type == "score":
            return x / (self.schedule.std(t)[:, None, None] + 1e-6)
        return x

## 20. Complete model and released loss

```text
conditions -> ContextEncoder
current state + normalized future -> VP noise future only
-> TrajectoryDiT -> x_start loss for ego + valid neighbors
```

This wrapper performs official initialization, constructs the unnoised current-state prefix, exposes the DPM-Solver-compatible denoiser callback, and computes the released separate ego/neighborhood loss terms.

**Dimension evolution:** Training I/O: conditions plus ego future `(B,80,4)` and neighbors `(B,10,80,4)` -> scalar loss dictionary.


In [ ]:
class DiffusionPlanner(nn.Module):
    """Official architecture with train and inference forward paths.

    Training inputs add ``sampled_trajectories`` `(B, 1+Pn, 1+T, 4)` and
    ``diffusion_time`` `(B,)`. Inference starts from a provided ``x_T`` through
    ``denoise`` so a DPM-Solver implementation can drive the same denoiser.
    """

    def __init__(self, config: PlannerConfig | None = None) -> None:
        super().__init__()
        self.config = config or PlannerConfig()
        self.encoder = ContextEncoder(self.config)
        self.decoder = TrajectoryDiT(self.config)
        self.schedule = VPSchedule()
        self.normalizer = StateNormalizer(self.config.participant_num)
        self._initialize_weights()

    def _initialize_weights(self) -> None:
        def initialize(module: nn.Module) -> None:
            if isinstance(module, nn.Linear):
                nn.init.xavier_uniform_(module.weight)
                if module.bias is not None:
                    nn.init.constant_(module.bias, 0)
            elif isinstance(module, nn.LayerNorm):
                nn.init.constant_(module.bias, 0)
                nn.init.constant_(module.weight, 1)
            elif isinstance(module, nn.Embedding):
                nn.init.normal_(module.weight, std=0.02)

        self.apply(initialize)
        nn.init.normal_(self.encoder.position.weight, std=0.02)
        nn.init.normal_(self.encoder.agents.type_emb.weight, std=0.02)
        nn.init.normal_(self.encoder.lanes.speed_limit_emb.weight, std=0.02)
        nn.init.normal_(self.encoder.lanes.traffic_emb.weight, std=0.02)
        nn.init.normal_(self.decoder.t_embedder.mlp[0].weight, std=0.02)
        nn.init.normal_(self.decoder.t_embedder.mlp[2].weight, std=0.02)
        for block in self.decoder.blocks:
            nn.init.constant_(block.adaLN_modulation[-1].weight, 0)
            nn.init.constant_(block.adaLN_modulation[-1].bias, 0)
        nn.init.constant_(self.decoder.final_layer.adaLN_modulation[-1].weight, 0)
        nn.init.constant_(self.decoder.final_layer.adaLN_modulation[-1].bias, 0)
        nn.init.constant_(self.decoder.final_layer.proj[-1].weight, 0)
        nn.init.constant_(self.decoder.final_layer.proj[-1].bias, 0)

    def _current_states(self, inputs: dict[str, torch.Tensor]) -> tuple[torch.Tensor, torch.Tensor]:
        ego = inputs["ego_current_state"][:, None, :4]
        neighbors = inputs["neighbor_agents_past"][:, : self.config.predicted_neighbor_num, -1, :4]
        neighbor_mask = neighbors.ne(0).any(dim=-1).logical_not()
        return torch.cat([ego, neighbors], dim=1), neighbor_mask

    def forward(
        self, inputs: dict[str, torch.Tensor]
    ) -> tuple[dict[str, torch.Tensor], dict[str, torch.Tensor]]:
        encoded = self.encoder(inputs)
        current, neighbor_mask = self._current_states(inputs)
        if self.training or ("sampled_trajectories" in inputs and "diffusion_time" in inputs):
            noisy = inputs["sampled_trajectories"].reshape(
                current.shape[0], self.config.participant_num, -1
            )
            score = self.decoder(
                noisy,
                inputs["diffusion_time"],
                encoded["encoding"],
                inputs["route_lanes"],
                neighbor_mask,
            )
            return encoded, {
                "score": score.reshape(current.shape[0], self.config.participant_num, -1, 4)
            }
        return encoded, {"current_states": current, "neighbor_current_mask": neighbor_mask}

    def denoise(
        self,
        x: torch.Tensor,
        time: torch.Tensor,
        encoded: dict[str, torch.Tensor],
        inputs: dict[str, torch.Tensor],
    ) -> torch.Tensor:
        """Denoiser callback for DPM-Solver or any inference integrator."""
        _, mask = self._current_states(inputs)
        return self.decoder(x, time, encoded["encoding"], inputs["route_lanes"], mask)

    def diffusion_loss(
        self,
        inputs: dict[str, torch.Tensor],
        ego_future: torch.Tensor,
        neighbor_future: torch.Tensor,
        neighbor_future_mask: torch.Tensor,
        eps: float = 1e-3,
    ) -> dict[str, torch.Tensor]:
        """Exact released loss construction, including current-state preservation."""
        batch, predicted_neighbors, horizon, _ = neighbor_future.shape
        if (
            predicted_neighbors != self.config.predicted_neighbor_num
            or horizon != self.config.future_len
        ):
            raise ValueError(
                "Future tensors do not match the configured official model dimensions."
            )
        current, current_mask = self._current_states(inputs)
        all_mask = torch.cat([current_mask.unsqueeze(-1), neighbor_future_mask], dim=-1)
        future = torch.cat([ego_future[:, None], neighbor_future], dim=1)
        clean = torch.cat([current[:, :, None], self.normalizer(future)], dim=2)
        clean[:, 1:][all_mask] = 0
        time = torch.rand(batch, device=clean.device) * (1 - eps) + eps
        mean, std = self.schedule.marginal_prob(clean[:, :, 1:], time)
        noisy = torch.cat([clean[:, :, :1], mean + std * torch.randn_like(mean)], dim=2)
        _, output = self({**inputs, "sampled_trajectories": noisy, "diffusion_time": time})
        score = output["score"][:, :, 1:]
        if self.config.diffusion_model_type == "score":
            loss = (score * std + (noisy[:, :, 1:] - mean) / std).square().sum(dim=-1)
        else:
            loss = (score - clean[:, :, 1:]).square().sum(dim=-1)
        valid_neighbors = ~neighbor_future_mask
        neighbor_loss = (
            loss[:, 1:][valid_neighbors].mean() if valid_neighbors.any() else loss.new_zeros(())
        )
        return {"ego_planning_loss": loss[:, 0].mean(), "neighbor_prediction_loss": neighbor_loss}

## 21. End-to-end default-dimension shape trace

This final cell creates official-shape synthetic inputs, executes the context encoder and denoiser, then computes the released loss. It is a shape and gradient check only, not a trained driving evaluation.


In [ ]:
config = PlannerConfig()
batch = 1
inputs = {
    "ego_current_state": torch.randn(batch, 10, device=device),
    "neighbor_agents_past": torch.randn(batch, 32, 21, 11, device=device),
    "static_objects": torch.randn(batch, 5, 10, device=device),
    "lanes": torch.randn(batch, 70, 20, 12, device=device),
    "lanes_speed_limit": torch.rand(batch, 70, 1, device=device),
    "lanes_has_speed_limit": torch.ones(batch, 70, 1, dtype=torch.bool, device=device),
    "route_lanes": torch.randn(batch, 25, 20, 12, device=device),
}
model = DiffusionPlanner(config).to(device).train()
context = model.encoder(inputs)
noisy = torch.randn(batch, 11, 324, device=device)
neighbor_mask = torch.zeros(batch, 10, dtype=torch.bool, device=device)
denoised = model.decoder(
    noisy,
    torch.full((batch,), 0.5, device=device),
    context["encoding"],
    inputs["route_lanes"],
    neighbor_mask,
)
losses = model.diffusion_loss(
    inputs,
    torch.randn(batch, 80, 4, device=device),
    torch.randn(batch, 10, 80, 4, device=device),
    torch.zeros(batch, 10, 80, dtype=torch.bool, device=device),
)
print("EXECUTED TENSOR LEDGER")
print("scene context tokens:", tuple(context["encoding"].shape))
print("route lanes input:", tuple(inputs["route_lanes"].shape))
print("noisy joint trajectory:", tuple(noisy.shape))
print("denoiser x_start:", tuple(denoised.shape))
print({name: float(value.detach()) for name, value in losses.items()})

## Optimization: learning from a synthetic batch

The two losses above are summed, then differentiated to update the randomly initialized model. This single synthetic step is a code probe, not evidence of planning quality. The official training recipe also uses augmentation, an EMA, and a learning-rate schedule.


In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
optimizer.zero_grad(set_to_none=True)
objective = losses["ego_planning_loss"] + losses["neighbor_prediction_loss"]
objective.backward()
grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
optimizer.step()
print("synthetic loss:", float(objective.detach()), "gradient norm:", float(grad_norm))

## A practical reverse-time example

A deterministic update uses the x-start prediction to estimate the noise at each time. The current state is copied into every iterate. This simple example is not the released DPM-Solver++ sampler, and its random model predictions are not a driving plan.


In [ ]:
model.eval()
with torch.no_grad():
    encoded, state = model(inputs)
    current = state["current_states"]
    x = torch.randn(batch, config.participant_num, config.future_len + 1, 4, device=device)
    x[:, :, 0] = current
    times = torch.linspace(1.0, 0.001, 4, device=device)
    for index, t in enumerate(times):
        time_batch = t.expand(batch)
        estimate = model.denoise(x.flatten(2), time_batch, encoded, inputs).reshape_as(x)
        estimate[:, :, 0] = current
        if index == len(times) - 1:
            x = estimate
            break
        alpha, sigma = model.schedule.marginal_prob(torch.ones_like(x[:, :, 1:]), time_batch)
        noise_estimate = (x[:, :, 1:] - alpha * estimate[:, :, 1:]) / sigma
        next_alpha, next_sigma = model.schedule.marginal_prob(
            torch.ones_like(x[:, :, 1:]), times[index + 1].expand(batch)
        )
        x = torch.cat(
            (current.unsqueeze(2), next_alpha * estimate[:, :, 1:] + next_sigma * noise_estimate),
            dim=2,
        )
    physical_futures = model.normalizer.inverse(x[:, :, 1:])
    assert torch.equal(x[:, :, 0], current)
    print("synthetic future shape:", tuple(physical_futures.shape))

## Diagnostic displacement and experiment recovery

ADE averages Euclidean position error over time; FDE measures the last position error. This example uses an arbitrary synthetic target and the untrained model, so these numbers are only an exercise in the calculation. They are not nuPlan closed-loop metrics. The package training command can save an epoch-boundary checkpoint containing model, optimizer, schedule, EMA and random states; a split/content and recipe identity is checked when resuming. Optional MLflow sessions and the CPU tuner are described in the README. Real data and accelerators are explicit external workflows.


In [ ]:
toy_target = torch.zeros_like(physical_futures[:, 0, :, :2])
distance = torch.linalg.vector_norm(physical_futures[:, 0, :, :2] - toy_target, dim=-1)
print("synthetic ADE, FDE in meters:", float(distance.mean()), float(distance[:, -1].mean()))